In [2]:
123

123

In [3]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

In [4]:
load_dotenv()   # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE','neo4j')    # 기본값 neo4j


In [5]:
# Neo4jGraph : Neo4j 연결정보를 받아서, 그래프 조회와 스키마 확인 기능을 하는 래퍼 클래스
graph = Neo4jGraph(
    url = NEO4J_URI,
    username= NEO4J_USERNAME,
    password= NEO4J_PASSWORD,
    database= NEO4J_DATABASE
)
print("Langchain과 Neo4j 연결 성공!")

Langchain과 Neo4j 연결 성공!


In [6]:
graph.refresh_schema() # 스키마 새로고침 (그래프 구조 변경시 실행)

print(graph.schema)     # 노드, 레이블, 속성, 관계 유형과 방향

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [7]:
query ="""
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id
ORDER BY student_id, course_id
""" 

result = graph.query(query)     # 쿼리를 받아 결과를 list(dict)로 반환 

result

[{'student_name': '홍길동',
  'course_name': 'Python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data Analysis',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': 'Database',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': 'Machine Learning',
  'student_id': 2,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'Python',
  'student_id': 3,
  'course_id': 101},
 {'student_name': '이민수',
  'course_name': 'Data Analysis',
  'student_id': 3,
  'course_id': 104},
 {'student_name': '박서연',
  'course_name': 'Machine Learning',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': 'Deep Learning',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': 'Database',
  'student_id': 5,
  'course_id': 102},
 {'student_name': '최준호',
  'course_name': 'Langchain',
  'student_id': 5,
  'course_id': 106}]

In [8]:
llm = ChatOpenAI(
    model= os.getenv('OPENAI_MODEL'),
    temperature= 0  # DB의 정보만 가졍ㄹ 것이므로 창의성은 0 (결정론적인 답변 = 일관적)
    )

In [12]:
# LLM과 Neo4j를 연결하여 자연어 질문을 Cypher 로 변환하고 답변하는 체인
chain = GraphCypherQAChain.from_llm(
    llm= llm,       # Cypher 생성 및 최종답변 llm
    graph = graph,  # 참조할 Neo4j 그래프 객체
    verbose= True,  # 로그 출력
    validate_cypher= True,  # 생성된 Cypher 검증
    return_intermediate_steps= True,     # 중간과정 함께 반환

    top_k= 10,      # 조회결과 10개
    allow_dangerous_requests = True     # DB 쿼리 실행 위험성 확인
)

In [13]:
question = 'Python 강의를 수강하는 학생을 알려줘.'

response = chain.invoke({'query': question})

response



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})
RETURN s;
Full Context:
[{'s': {'name': '이민수', 'student_id': 3, 'age': 24}}, {'s': {'name': '홍길동', 'student_id': 1, 'age': 26}}]

> Finished chain.


{'query': 'Python 강의를 수강하는 학생을 알려줘.',
 'result': 'Python 강의를 수강하는 학생을 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;"},
  {'context': [{'s': {'name': '이민수', 'student_id': 3, 'age': 24}},
    {'s': {'name': '홍길동', 'student_id': 1, 'age': 26}}]}]}

In [ ]:
response['intermediate_steps']  # 중간과정 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;"},
 {'context': [{'s': {'name': '이민수', 'student_id': 3, 'age': 24}},
   {'s': {'name': '홍길동', 'student_id': 1, 'age': 26}}]}]

In [15]:
# 질문 답변 chain 함수
def ask_graph(question: str) -> dict:
    if not question.strip():
        raise ValueError('질문을 입력하셔야 합니다!!')
    
    response = chain.invoke({'query': question})
    
    print(f'[질문] {question}')
    print(f'[최종답변] {response['result']}')
    
    # 중간과정 추가시
    for step in response.get('intermediate_steps', []):
        if 'query' in step:
            print(f'[생성된 Cypher] {step['query']}')
            
        if 'context' in step:
            print(f'[조회 결과] {step['context']}')
            
    return response

In [18]:
ask_graph('Tell me about the lectures 홍길동 took and the instructor in charge')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS lecture, i.name AS instructor;

Full Context:
[{'lecture': 'Python', 'instructor': 'Capybara'}, {'lecture': 'Data Analysis', 'instructor': 'Alice'}]

> Finished chain.
[질문] Tell me about the lectures 홍길동 took and the instructor in charge
[최종답변] 홍길동은 Python과 Data Analysis 강의를 수강했으며, 각각 Capybara와 Alice가 담당 강사입니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS lecture, i.name AS instructor;

[조회 결과] [{'lecture': 'Python', 'instructor': 'Capybara'}, {'lecture': 'Data Analysis', 'instructor': 'Alice'}]


{'query': 'Tell me about the lectures 홍길동 took and the instructor in charge',
 'result': '홍길동은 Python과 Data Analysis 강의를 수강했으며, 각각 Capybara와 Alice가 담당 강사입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN c.name AS lecture, i.name AS instructor;\n"},
  {'context': [{'lecture': 'Python', 'instructor': 'Capybara'},
    {'lecture': 'Data Analysis', 'instructor': 'Alice'}]}]}

In [20]:
ask_graph('인공지능 카테코리에 속한 강의를 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c.name, c.course_id, c.level, c.duration;

Full Context:
[{'c.name': 'Machine Learning', 'c.course_id': 103, 'c.level': '입문', 'c.duration': 48}, {'c.name': 'Deep Learning', 'c.course_id': 105, 'c.level': '중급', 'c.duration': 52}, {'c.name': 'Langchain', 'c.course_id': 106, 'c.level': '중급', 'c.duration': 32}]

> Finished chain.
[질문] 인공지능 카테코리에 속한 강의를 알려줘
[최종답변] 인공지능 카테고리에 속한 강의는 다음과 같습니다.

- Machine Learning (입문, 48시간)
- Deep Learning (중급, 52시간)
- Langchain (중급, 32시간)
[생성된 Cypher] MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c.name, c.course_id, c.level, c.duration;

[조회 결과] [{'c.name': 'Machine Learning', 'c.course_id': 103, 'c.level': '입문', 'c.duration': 48}, {'c.name': 'Deep Learning', 'c.course_id': 105, 'c.level': '중급', 'c.duration': 52}, {'c.name': 'Langchain', 'c.course_id': 106, 'c.level': '중급', 'c.duration': 32}]


{'query': '인공지능 카테코리에 속한 강의를 알려줘',
 'result': '인공지능 카테고리에 속한 강의는 다음과 같습니다.\n\n- Machine Learning (입문, 48시간)\n- Deep Learning (중급, 52시간)\n- Langchain (중급, 32시간)',
 'intermediate_steps': [{'query': "MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})\nRETURN c.name, c.course_id, c.level, c.duration;\n"},
  {'context': [{'c.name': 'Machine Learning',
     'c.course_id': 103,
     'c.level': '입문',
     'c.duration': 48},
    {'c.name': 'Deep Learning',
     'c.course_id': 105,
     'c.level': '중급',
     'c.duration': 52},
    {'c.name': 'Langchain',
     'c.course_id': 106,
     'c.level': '중급',
     'c.duration': 32}]}]}

In [21]:
ask_graph('수강생이 가장 많은 강의를 알려줘.')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
Full Context:
[{'course_name': 'Database', 'student_count': 2}]

> Finished chain.
[질문] 수강생이 가장 많은 강의를 알려줘.
[최종답변] 수강생이 가장 많은 강의는 **Database**이며, 수강생은 **2명**입니다.
[생성된 Cypher] MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
[조회 결과] [{'course_name': 'Database', 'student_count': 2}]


{'query': '수강생이 가장 많은 강의를 알려줘.',
 'result': '수강생이 가장 많은 강의는 **Database**이며, 수강생은 **2명**입니다.',
 'intermediate_steps': [{'query': 'MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nRETURN c.name AS course_name, count(s) AS student_count\nORDER BY student_count DESC\nLIMIT 1'},
  {'context': [{'course_name': 'Database', 'student_count': 2}]}]}

In [24]:
ask_graph('Tell me about another student who is taking the same course as 홍길동')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> s.student_id
RETURN other, c;
Full Context:
[{'other': {'name': '이민수', 'student_id': 3, 'age': 24}, 'c': {'duration': 40, 'course_id': 101, 'level': '초급', 'name': 'Python'}}, {'other': {'name': '이민수', 'student_id': 3, 'age': 24}, 'c': {'duration': 40, 'course_id': 104, 'level': '입문', 'name': 'Data Analysis'}}]

> Finished chain.
[질문] Tell me about another student who is taking the same course as 홍길동
[최종답변] 이민수 학생이 홍길동 학생과 같은 과정을 수강하고 있습니다. 이민수 학생은 Python과 Data Analysis 과정을 수강 중입니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> s.student_id
RETURN other, c;
[조회 결과] [{'other': {'name': '이민수', 'student_id': 3, 'age': 24}, 'c': {'duration': 40, 'course_id': 101, 'level': '초급', 'name': 'Python'}}, {'other': {'name': '이민수', 'student

{'query': 'Tell me about another student who is taking the same course as 홍길동',
 'result': '이민수 학생이 홍길동 학생과 같은 과정을 수강하고 있습니다. 이민수 학생은 Python과 Data Analysis 과정을 수강 중입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:ENROLLED_IN]-(other:Student)\nWHERE other.student_id <> s.student_id\nRETURN other, c;"},
  {'context': [{'other': {'name': '이민수', 'student_id': 3, 'age': 24},
     'c': {'duration': 40, 'course_id': 101, 'level': '초급', 'name': 'Python'}},
    {'other': {'name': '이민수', 'student_id': 3, 'age': 24},
     'c': {'duration': 40,
      'course_id': 104,
      'level': '입문',
      'name': 'Data Analysis'}}]}]}